# Module 04 — UDF Performance

Serialization cost: Python UDF vs Pandas UDF vs native Spark.

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import get_spark
from shared import config
from shared.utils import timed_action

spark = get_spark('module-04-udfs')
trips = spark.read.parquet(*config.yellow_taxi_paths())
df = trips.select('PULocationID').filter('PULocationID IS NOT NULL').limit(1_000_000)

## 1. Vanilla Python UDF

Row-by-row Python callback — highest serialization overhead.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import hashlib

@udf(StringType())
def py_hash(val):
    if val is None:
        return None
    return hashlib.md5(str(val).encode()).hexdigest()

result_py = df.withColumn('h', py_hash('PULocationID'))
_, t_py = timed_action(result_py)
print(f'Python UDF: {t_py:.2f}s')

## 2. Pandas UDF (Arrow)

Batch transfer via Apache Arrow — much lower per-row overhead.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(StringType())
def pandas_hash(series: pd.Series) -> pd.Series:
    return series.astype(str).apply(lambda x: hashlib.md5(x.encode()).hexdigest())

result_pandas = df.withColumn('h', pandas_hash('PULocationID'))
_, t_pandas = timed_action(result_pandas)
print(f'Pandas UDF: {t_pandas:.2f}s')

## 3. Native Spark SQL

No Python boundary — fastest.

In [ ]:
from pyspark.sql.functions import sha2, col

result_native = df.withColumn('h', sha2(col('PULocationID').cast('string'), 256))
_, t_native = timed_action(result_native)
print(f'Native Spark: {t_native:.2f}s')

In [ ]:
print(f'\nSpeedup vs Python UDF:')
print(f'  Pandas UDF:  {t_py/t_pandas:.1f}x')
print(f'  Native:      {t_py/t_native:.1f}x')

## Why?

| Approach | Data path | Cost |
|----------|-----------|------|
| Python UDF | JVM → pickle → Python (per row) | Highest |
| Pandas UDF | JVM → Arrow batches → Python | Medium |
| Native | JVM only | Lowest |

Check Spark UI **Stages** tab: Python UDF shows `PythonUDF` in the plan.

## Key Takeaways

1. Prefer native Spark functions whenever possible.
2. Pandas UDFs are the best choice when custom Python logic is required.
3. Vanilla Python UDFs are a last resort for performance-critical paths.